# Vector DB Specialisation — FAISS vs Chroma vs Pinecone vs Qdrant

We index the **same document, the same chunks, the same embeddings** into four different vector stores, then run the **same query** against all four. The goal isn't to find a "winner" — it's to see, hands-on, what actually changes between an in-memory index (FAISS), a local persistent store (Chroma), a fully managed cloud service (Pinecone), and a self-host-or-cloud store with more retrieval flexibility (Qdrant).

- `chroma.ipynb`, `pinecone.ipynb`, `qdrant.ipynb` in this same folder are the original single-DB notebooks this is built from
- This notebook reuses their exact code patterns, just side by side, plus a timing/comparison pass at the end

## ⚠️ Before you run this: API keys

- **FAISS** and **Chroma** need nothing extra — they run locally.
- **Pinecone** needs `PINECONE_API_KEY` in your `.env` (free tier at pinecone.io).
- **Qdrant** needs `QDRANT_URL` and `QDRANT_API_KEY` in your `.env` (free cluster at cloud.qdrant.io).

If a key is missing, that section will print a warning and skip itself instead of crashing the whole notebook — you'll still see FAISS/Chroma results either way.

> 🔒 **Security note**: the existing `pinecone.ipynb` and `qdrant.ipynb` in this folder have real API keys/tokens hardcoded directly in the cells. If you ever share or commit those files, rotate those keys first — this notebook deliberately reads credentials from `.env` instead so it's safe to share.

In [1]:
from pathlib import Path
import sys, os

if "__vsc_ipynb_file__" in globals():
    start_path = Path(__vsc_ipynb_file__).resolve().parent
else:
    start_path = Path.cwd().resolve()

project_root = start_path
for candidate in [start_path, *start_path.parents]:
    if (candidate / "src").exists():
        project_root = candidate
        break

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Settings() loads .env relative to cwd, and .env lives in src/backend/ in this
# project (not the project root) -- so we chdir there specifically for the import,
# but use absolute paths (below) for any data/persistence folders, since those
# live under src/VectorDBs/, a different folder entirely.
backend_dir = project_root / "src" / "backend"
vectordb_dir = project_root / "src" / "VectorDBs"
os.chdir(backend_dir)

from src.backend.core.config import settings

# pydantic-settings parses .env for its OWN declared fields only (OPENAI_MODEL,
# OPENAI_API_KEY, etc.) -- it does NOT copy those values into os.environ as a side
# effect. PINECONE_API_KEY / QDRANT_URL / QDRANT_API_KEY aren't modeled in Settings
# at all, so we load_dotenv() explicitly to get them into os.environ for os.getenv().
from dotenv import load_dotenv
load_dotenv(backend_dir / ".env")

print(f"Project root:   {project_root}")
print(f"OPENAI_API_KEY loaded: {bool(settings.OPENAI_API_KEY)}")

if not settings.OPENAI_API_KEY:
    raise RuntimeError("OPENAI_API_KEY missing -- check src/backend/.env")

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY", "")
QDRANT_URL = os.getenv("QDRANT_URL", "")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")

print(f"PINECONE_API_KEY found: {bool(PINECONE_API_KEY)}")
print(f"QDRANT_URL + QDRANT_API_KEY found: {bool(QDRANT_URL and QDRANT_API_KEY)}")

Project root:   D:\Mentoring\learwithsarvesh\AI Engineer Ready\WEEK 7 — PROJECT 1 (Part 1) Production-Grade RAG System\Live
OPENAI_API_KEY loaded: True
PINECONE_API_KEY found: True
QDRANT_URL + QDRANT_API_KEY found: True


## Shared setup: one document, one set of chunks, one embedding model

Everything below indexes the exact same `Acme_FY2024_UltraDense_Report.pdf` (already in `TempData/`, same file the original `chroma.ipynb`/`pinecone.ipynb`/`qdrant.ipynb` use) so the comparison is fair -- any difference we see is the vector DB, not the data.

In [2]:
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from src.backend.logger import GLOBAL_LOGGER as log
from src.backend.rag.embeddings import get_embeddings


def load_document(directory_path):
    documents = []
    for filename in os.listdir(directory_path):
        file_path = os.path.join(directory_path, filename)
        if filename.endswith(".pdf"):
            documents.extend(PyPDFLoader(file_path).load())
        elif filename.endswith(".docx"):
            documents.extend(Docx2txtLoader(file_path).load())
        elif filename.endswith(".txt"):
            documents.extend(TextLoader(file_path, encoding="utf-8").load())
    return documents


data_dir = vectordb_dir / "TempData"
docs = load_document(str(data_dir))

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=50)
chunks = splitter.split_documents(docs)

embeddings = get_embeddings("OpenAI (text-embedding-3-small)")
embedding_dim = len(embeddings.embed_query("check embedding size"))

QUERY = "What is the revenue increase in FY24?"

print(f"Loaded {len(docs)} page(s), split into {len(chunks)} chunk(s)")
print(f"Embedding dimension: {embedding_dim}")
print(f"Shared test query: {QUERY!r}")

d:\Mentoring\learwithsarvesh\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Loaded 2 page(s), split into 6 chunk(s)

Embedding dimension: 1536

Shared test query: 'What is the revenue increase in FY24?'

---
## 1. FAISS — the baseline you already know

- In-memory, no server, no account
- Fastest to spin up, but persistence is manual (`save_local` / `load_local`) and there's no built-in multi-machine scaling
- This is exactly `faiss_store.py` in the main project, just inline here for comparison

In [3]:
import time
from langchain_community.vectorstores import FAISS

t0 = time.perf_counter()
faiss_store = FAISS.from_documents(chunks, embeddings)
faiss_index_time = time.perf_counter() - t0

t0 = time.perf_counter()
faiss_results = faiss_store.similarity_search(QUERY, k=1)
faiss_query_time = time.perf_counter() - t0

print(f"Indexed in {faiss_index_time:.2f}s, queried in {faiss_query_time:.3f}s")
print(faiss_results[0].page_content[:300])

HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Loading faiss with AVX2 support.


Successfully loaded faiss with AVX2 support.


HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Indexed in 0.89s, queried in 0.407s

through automation, cost optimization initiatives, and deleveraging of the balance sheet. Management focused on strengthening
internal controls, improving working capital efficiency, and mitigating interest rate and foreign exchange risks.
FINANCIAL HIGHLIGHTS: Revenue increased to USD 120.0 million

---
## 2. Chroma — local, persistent, zero-ops

- Runs in-process like FAISS, but **persists automatically** to a SQLite-backed folder on disk -- no manual save/load step
- Still single-machine, no separate server or account needed
- Good middle ground: more durable than FAISS, simpler than a managed cloud service

In [4]:
import chromadb
from langchain_chroma import Chroma

chroma_persist_dir = str(vectordb_dir / "chroma_db")
_chroma_client = chromadb.PersistentClient(path=chroma_persist_dir)

chroma_store = Chroma(
    collection_name="vectordb_comparison",
    embedding_function=embeddings,
    persist_directory=chroma_persist_dir,
)

t0 = time.perf_counter()
chroma_store.add_documents(documents=chunks)
chroma_index_time = time.perf_counter() - t0

t0 = time.perf_counter()
chroma_results = chroma_store.similarity_search(QUERY, k=1)
chroma_query_time = time.perf_counter() - t0

print(f"Indexed in {chroma_index_time:.2f}s, queried in {chroma_query_time:.3f}s")
print(f"Persisted to: {chroma_persist_dir}")
print(chroma_results[0].page_content[:300])

Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Indexed in 0.61s, queried in 0.370s

Persisted to: D:\Mentoring\learwithsarvesh\AI Engineer Ready\WEEK 7 — PROJECT 1 (Part 1) Production-Grade RAG System\Live\src\VectorDBs\chroma_db

through automation, cost optimization initiatives, and deleveraging of the balance sheet. Management focused on strengthening
internal controls, improving working capital efficiency, and mitigating interest rate and foreign exchange risks.
FINANCIAL HIGHLIGHTS: Revenue increased to USD 120.0 million

---
## 3. Pinecone — fully managed, serverless cloud

- No server to run yourself -- you call an API, Pinecone hosts the index
- Scales automatically; you pay for what you store/query, not for idle infrastructure
- The trade-off: every operation is a network call, so expect higher latency than FAISS/Chroma running on your own machine
- Skips automatically if `PINECONE_API_KEY` isn't set

In [5]:
pinecone_results = None
pinecone_index_time = pinecone_query_time = None

if not PINECONE_API_KEY:
    print("Skipping Pinecone -- PINECONE_API_KEY not set in .env")
else:
    try:
        from pinecone import Pinecone, ServerlessSpec
        from langchain_pinecone import PineconeVectorStore

        pc = Pinecone(api_key=PINECONE_API_KEY)
        index_name = "vectordb-comparison"

        if not pc.has_index(index_name):
            pc.create_index(
                name=index_name,
                dimension=embedding_dim,
                metric="cosine",
                spec=ServerlessSpec(cloud="aws", region="us-east-1"),
            )
        pinecone_index = pc.Index(index_name)

        pinecone_store = PineconeVectorStore(index=pinecone_index, embedding=embeddings)

        t0 = time.perf_counter()
        pinecone_store.add_documents(documents=chunks)
        pinecone_index_time = time.perf_counter() - t0

        t0 = time.perf_counter()
        pinecone_results = pinecone_store.similarity_search(QUERY, k=1)
        pinecone_query_time = time.perf_counter() - t0

        print(f"Indexed in {pinecone_index_time:.2f}s, queried in {pinecone_query_time:.3f}s")
        print(pinecone_results[0].page_content[:300])
    except Exception as e:
        print(f"Skipping Pinecone -- couldn't reach the index ({type(e).__name__}: {e})")

HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Indexed in 2.55s, queried in 0.623s

through automation, cost optimization initiatives, and deleveraging of the balance sheet. Management focused on strengthening
internal controls, improving working capital efficiency, and mitigating interest rate and foreign exchange risks.
FINANCIAL HIGHLIGHTS: Revenue increased to USD 120.0 million

---
## 4. Qdrant — self-host or managed cloud, more retrieval flexibility

- Same managed-cloud convenience as Pinecone here, but Qdrant can also be **self-hosted** (Docker, your own servers) if you need full control
- Supports hybrid search (dense + sparse vectors) natively -- useful if you ever need keyword + semantic search combined
- Skips automatically if `QDRANT_URL` / `QDRANT_API_KEY` aren't set

In [6]:
qdrant_results = None
qdrant_index_time = qdrant_query_time = None

if not (QDRANT_URL and QDRANT_API_KEY):
    print("Skipping Qdrant -- QDRANT_URL / QDRANT_API_KEY not set in .env")
else:
    try:
        from qdrant_client import QdrantClient
        from langchain_qdrant import QdrantVectorStore

        qdrant_client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
        collection_name = "vectordb-comparison"

        if not qdrant_client.collection_exists(collection_name=collection_name):
            qdrant_client.create_collection(
                collection_name,
                vectors_config={"size": embedding_dim, "distance": "Cosine"},
            )

        qdrant_store = QdrantVectorStore(
            client=qdrant_client,
            collection_name=collection_name,
            embedding=embeddings,
        )

        t0 = time.perf_counter()
        qdrant_store.add_documents(documents=chunks)
        qdrant_index_time = time.perf_counter() - t0

        t0 = time.perf_counter()
        qdrant_results = qdrant_store.similarity_search(QUERY, k=1)
        qdrant_query_time = time.perf_counter() - t0

        print(f"Indexed in {qdrant_index_time:.2f}s, queried in {qdrant_query_time:.3f}s")
        print(qdrant_results[0].page_content[:300])
    except Exception as e:
        # Free-tier Qdrant clusters get paused after inactivity -- a dead URL
        # shouldn't take down the rest of the comparison.
        print(f"Skipping Qdrant -- couldn't reach the cluster ({type(e).__name__}: {e})")
        print("If you're using a free Qdrant Cloud cluster, check it hasn't been paused.")

HTTP Request: GET https://c72720c9-8267-4e2b-91cc-e2ce70a87e6a.sa-east-1-0.aws.cloud.qdrant.io:6333/collections/vectordb-comparison/exists "HTTP/1.1 404 Not Found"


Skipping Qdrant -- couldn't reach the cluster (UnexpectedResponse: Unexpected Response: 404 (Not Found)
Raw response content:
b'404 page not found\n')

HTTP Request: GET https://c72720c9-8267-4e2b-91cc-e2ce70a87e6a.sa-east-1-0.aws.cloud.qdrant.io:6333 "HTTP/1.1 404 Not Found"


If you're using a free Qdrant Cloud cluster, check it hasn't been paused.

d:\Mentoring\learwithsarvesh\.venv\lib\site-packages\qdrant_client\qdrant_remote.py:275: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(


---
## Side-by-side comparison

Same query, four stores. Run this after running whichever sections above had credentials available.

In [7]:
rows = [
    ("FAISS", faiss_index_time, faiss_query_time, faiss_results[0].page_content[:120]),
    ("Chroma", chroma_index_time, chroma_query_time, chroma_results[0].page_content[:120]),
]
if pinecone_results:
    rows.append(("Pinecone", pinecone_index_time, pinecone_query_time, pinecone_results[0].page_content[:120]))
if qdrant_results:
    rows.append(("Qdrant", qdrant_index_time, qdrant_query_time, qdrant_results[0].page_content[:120]))

print(f"{'Store':<10} {'Index (s)':<12} {'Query (s)':<12} Top result preview")
print("-" * 90)
for name, idx_t, qry_t, preview in rows:
    print(f"{name:<10} {idx_t:<12.2f} {qry_t:<12.3f} {preview}...")

Store      Index (s)    Query (s)    Top result preview

------------------------------------------------------------------------------------------

FAISS      0.89         0.407        through automation, cost optimization initiatives, and deleveraging of the balance sheet. Management focused on strength...

Chroma     0.61         0.370        through automation, cost optimization initiatives, and deleveraging of the balance sheet. Management focused on strength...

Pinecone   2.55         0.623        through automation, cost optimization initiatives, and deleveraging of the balance sheet. Management focused on strength...

**What to actually look at in those numbers:**
- Index/query time for FAISS and Chroma should be close -- both run locally, in-process
- Pinecone/Qdrant query times include a real network round-trip, so they're usually slower per-call than FAISS/Chroma, even though the underlying search itself may be just as fast or faster at scale
- The result *content* should be near-identical across all four -- same chunks, same embeddings, same similarity metric. If one store returns a wildly different top result, that's worth double-checking (different distance metric, different chunking applied, etc.)

---
## Summary: when to use which

| | FAISS | Chroma | Pinecone | Qdrant |
|---|---|---|---|---|
| Hosting | In-memory / local | Local disk (SQLite) | Managed cloud (serverless) | Self-host **or** managed cloud |
| Persistence | Manual `save_local`/`load_local` | Automatic | Automatic | Automatic |
| Setup | `pip install faiss-cpu`, nothing else | `pip install chromadb`, nothing else | Account + API key | Account + API key, or your own Docker container |
| Scaling | Manual, RAM-bound, single machine | Single machine | Auto-scaling, fully managed | Horizontal, you configure it (or let the cloud tier manage it) |
| Best for | Notebooks, prototyping, small fixed datasets | Small-to-mid production, simple ops | Production at scale, zero infra to manage | Production needing self-host control, or hybrid (dense+sparse) search |

**Rule of thumb**: start with FAISS or Chroma while building. Move to Pinecone if you want zero infrastructure to manage. Move to Qdrant if you need to self-host, need hybrid search, or want more control over the deployment than Pinecone gives you.